Create a Spark Session


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, min, max, count

In [2]:
spark = SparkSession.builder.appName("Spark_Assignment").getOrCreate()

Read csv file


In [3]:
df = spark.read.csv("employees.csv", header=True, inferSchema=True)

In [4]:
df.show()

+----------+-------+----+----------+------+------+
|EmployeeID|   Name| Age|Department|Salary|Region|
+----------+-------+----+----------+------+------+
|       101|  Alice|  25|        HR| 45000| North|
|       102|    Bob|  30|        IT| 60000| South|
|       103|Charlie|  35|   Finance| 70000|  East|
|       104|  David|  28|        IT| 55000|  West|
|       105|    Eva|  32|        HR| 50000| North|
|       106|  Frank|NULL|   Finance| 65000| South|
|       107|  Grace|  29|        IT|  NULL|  West|
|       108|  Helen|  25|        HR| 45000| North|
|       108|  Helen|  25|        HR| 45000| North|
|       109|    Ian|  40|   Finance| 80000|  East|
|       110|   Jack|  27|        IT| 52000| South|
+----------+-------+----+----------+------+------+



In [5]:
df.columns

['EmployeeID', 'Name', 'Age', 'Department', 'Salary', 'Region']

In [6]:
df.dtypes

[('EmployeeID', 'int'),
 ('Name', 'string'),
 ('Age', 'int'),
 ('Department', 'string'),
 ('Salary', 'int'),
 ('Region', 'string')]

Data Cleaning

1. Remove Duplicate rows


In [7]:
df.dropDuplicates()

DataFrame[EmployeeID: int, Name: string, Age: int, Department: string, Salary: int, Region: string]

2. Handling missing values: Replace with mean value

In [8]:
from pyspark.sql.functions import mean

mean_salary = df.select(mean("Salary")).collect()[0][0]
mean_age = df.select(mean("Age")).collect()[0][0]

df = df.fillna({"Salary": mean_salary})
df = df.fillna({"Age": mean_age})

In [9]:
df.show()

+----------+-------+---+----------+------+------+
|EmployeeID|   Name|Age|Department|Salary|Region|
+----------+-------+---+----------+------+------+
|       101|  Alice| 25|        HR| 45000| North|
|       102|    Bob| 30|        IT| 60000| South|
|       103|Charlie| 35|   Finance| 70000|  East|
|       104|  David| 28|        IT| 55000|  West|
|       105|    Eva| 32|        HR| 50000| North|
|       106|  Frank| 29|   Finance| 65000| South|
|       107|  Grace| 29|        IT| 56700|  West|
|       108|  Helen| 25|        HR| 45000| North|
|       108|  Helen| 25|        HR| 45000| North|
|       109|    Ian| 40|   Finance| 80000|  East|
|       110|   Jack| 27|        IT| 52000| South|
+----------+-------+---+----------+------+------+



  Check for incorrect or inconsistent data


1. Check data types

In [10]:
df.printSchema()

root
 |-- EmployeeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Region: string (nullable = true)



2.Check unique values in categorical columns

In [11]:
df.select("Department").distinct().show()

+----------+
|Department|
+----------+
|        HR|
|   Finance|
|        IT|
+----------+



Filter Data


1. Filter by age

In [13]:
age_df = df.filter(df["Age"] > 30)
age_df.show()

+----------+-------+---+----------+------+------+
|EmployeeID|   Name|Age|Department|Salary|Region|
+----------+-------+---+----------+------+------+
|       103|Charlie| 35|   Finance| 70000|  East|
|       105|    Eva| 32|        HR| 50000| North|
|       109|    Ian| 40|   Finance| 80000|  East|
+----------+-------+---+----------+------+------+



2. Filter by category

In [14]:
hr_df = df.filter(df["Department"] == "IT")
hr_df.show()

+----------+-----+---+----------+------+------+
|EmployeeID| Name|Age|Department|Salary|Region|
+----------+-----+---+----------+------+------+
|       102|  Bob| 30|        IT| 60000| South|
|       104|David| 28|        IT| 55000|  West|
|       107|Grace| 29|        IT| 56700|  West|
|       110| Jack| 27|        IT| 52000| South|
+----------+-----+---+----------+------+------+



3. Filter by region

In [15]:
south_df = df.filter(df["Region"] == 'South')
south_df.show()

+----------+-----+---+----------+------+------+
|EmployeeID| Name|Age|Department|Salary|Region|
+----------+-----+---+----------+------+------+
|       102|  Bob| 30|        IT| 60000| South|
|       106|Frank| 29|   Finance| 65000| South|
|       110| Jack| 27|        IT| 52000| South|
+----------+-----+---+----------+------+------+



Transform Column

1 Change Age to integer

In [16]:
df = df.withColumn("Age", col("Age").cast("integer"))

Aggregation

1. Count total rows

In [17]:
print(df.count())

11


2. Average Salary

In [18]:
df.select(avg("Salary").alias("Average Salary"),).show()

+--------------+
|Average Salary|
+--------------+
|       56700.0|
+--------------+



3. Minimum and Maximum Age

In [19]:
df.select(
    min("Age").alias("Minimum Age"),
    max("Age").alias("Maximum Age")
).show()

+-----------+-----------+
|Minimum Age|Maximum Age|
+-----------+-----------+
|         25|         40|
+-----------+-----------+



Group Data

1. count()

In [28]:
df.groupBy("Department").agg(count("*").alias("Employees")).show()

+----------+---------+
|Department|Employees|
+----------+---------+
|        HR|        4|
|   Finance|        3|
|        IT|        4|
+----------+---------+



2. avg()

In [26]:
df.groupBy("Department").avg("Salary").alias("Average Salary").show()

+----------+-----------------+
|Department|      avg(Salary)|
+----------+-----------------+
|        HR|          46250.0|
|   Finance|71666.66666666667|
|        IT|          55925.0|
+----------+-----------------+



3. sum()

In [25]:
df.groupBy("Department").sum("Salary").alias("Total Salary").show()

+----------+-----------+
|Department|sum(Salary)|
+----------+-----------+
|        HR|     185000|
|   Finance|     215000|
|        IT|     223700|
+----------+-----------+



Simple Pipeline

In [29]:
pipeline = (
    spark.read.csv("employees.csv",
                   header=True,
                   inferSchema=True)
    .dropDuplicates()
    .na.drop(subset=["Age", "Salary"])
    .filter(col("Age") > 25)
    .withColumnRenamed("Salary", "MonthlySalary")
)

pipeline.show()

+----------+-------+---+----------+-------------+------+
|EmployeeID|   Name|Age|Department|MonthlySalary|Region|
+----------+-------+---+----------+-------------+------+
|       104|  David| 28|        IT|        55000|  West|
|       105|    Eva| 32|        HR|        50000| North|
|       103|Charlie| 35|   Finance|        70000|  East|
|       109|    Ian| 40|   Finance|        80000|  East|
|       102|    Bob| 30|        IT|        60000| South|
|       110|   Jack| 27|        IT|        52000| South|
+----------+-------+---+----------+-------------+------+



In [30]:
pipeline.select(avg("MonthlySalary")).show()

+------------------+
|avg(MonthlySalary)|
+------------------+
|61166.666666666664|
+------------------+



In [32]:
dept_summary = df.groupBy("Department").agg(
    count("*").alias("Employees"),
    avg("Salary").alias("Average Salary"),
    sum("Salary").alias("Total Salary")
)

dept_summary.show()

+----------+---------+-----------------+------------+
|Department|Employees|   Average Salary|Total Salary|
+----------+---------+-----------------+------------+
|        HR|        4|          46250.0|      185000|
|   Finance|        3|71666.66666666667|      215000|
|        IT|        4|          55925.0|      223700|
+----------+---------+-----------------+------------+



In [33]:
dept_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("department_summary.csv")